In [644]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import accuracy_score, classification_report

# Suport functions

In [645]:
def confusion(true, pred):
    """
    Function for pretty printing confusion matrices
    """
    true.name = 'target'
    pred.name = 'predicted'
    cm = pd.crosstab(true.reset_index(drop=True), pred.reset_index(drop=True))
    cm = cm[cm.index]
    return cm

# Data loading

In [ ]:
ILDS = pd.read_csv("log_ILDS_train_X_no_resampling.csv", delimiter=',', header=None)


ILDS.columns = ['Age','TP','BD','AR','DBratio','logTB','logDB','logAlkphos','logSgpt','logSgot','Female', 'Target']

ILDS.shape

(440, 12)

In [647]:
X = ILDS.loc[:, ILDS.columns != 'Target']
y = ILDS['Target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, stratify = y, random_state=1234)

In [648]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state=42)
X_train, y_train = rus.fit_resample(X_train, y_train)


In [649]:
results_df = pd.DataFrame(index=[], columns= ['Accuracy', 'F1 Macro', 'Precision Macro', 'Recall Macro'])

# Random forest

In [650]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)


In [651]:
confusion(y_train, pd.Series(rf.predict(X_train)))

predicted,0,1
target,,
0,87,0
1,0,87


In [652]:
confusion(y_test, pd.Series(rf.predict(X_test)))     

predicted,0,1
target,,
0,81,22
1,18,25


In [653]:
cross_val_results = pd.DataFrame(cross_validate(rf , X_train, y_train, cv = 5, 
                            scoring = ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro'] ))

results_df.loc['RF',:] = cross_val_results[['test_accuracy', 'test_f1_macro',
       'test_precision_macro', 'test_recall_macro']].mean().values
results_df

,Accuracy,F1 Macro,Precision Macro,Recall Macro
RF,0.621345,0.619199,0.625543,0.622222


# Gradient Boosting

In [654]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight

weights = compute_sample_weight(class_weight='balanced', y=y_train)
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train, y_train, sample_weight=weights)
gb_pred = gb.predict(X_test)

In [655]:
cross_val_results = pd.DataFrame(cross_validate(gb , X_train, y_train, cv = 5, 
                            scoring = ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro'] ))

results_df.loc['GB',:] = cross_val_results[['test_accuracy', 'test_f1_macro',
       'test_precision_macro', 'test_recall_macro']].mean().values
results_df

,Accuracy,F1 Macro,Precision Macro,Recall Macro
RF,0.621345,0.619199,0.625543,0.622222
GB,0.598487,0.596573,0.601339,0.598693


# Voting classifier

In [656]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier

log_clf = LogisticRegression(class_weight="balanced")
svc_clf = SVC(probability=True, class_weight="balanced")
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
voting = VotingClassifier(estimators=[
    ('lr', log_clf), ('svc', svc_clf), ('rf', rf)
], voting='soft')

voting.fit(X_train, y_train)
voting_pred = voting.predict(X_test)


In [657]:
confusion(y_train, pd.Series(voting.predict(X_train)))

predicted,0,1
target,,
0,79,8
1,4,83


In [658]:
confusion(y_test, pd.Series(voting.predict(X_test)))

predicted,0,1
target,,
0,75,28
1,14,29


In [659]:
cross_val_results = pd.DataFrame(cross_validate(voting , X_train, y_train, cv = 5, 
                            scoring = ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro'] ))

results_df.loc['CLF',:] = cross_val_results[['test_accuracy', 'test_f1_macro',
       'test_precision_macro', 'test_recall_macro']].mean().values
results_df

,Accuracy,F1 Macro,Precision Macro,Recall Macro
RF,0.621345,0.619199,0.625543,0.622222
GB,0.598487,0.596573,0.601339,0.598693
CLF,0.672605,0.668591,0.681769,0.672222


# XGBoost

In [660]:
from xgboost import XGBClassifier

xgb = XGBClassifier(scale_pos_weight=1/3)
xgb.fit(X_train, y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [661]:
cross_val_results = pd.DataFrame(cross_validate(xgb , X_train, y_train, cv = 5, 
                            scoring = ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro'] ))

results_df.loc['XGB',:] = cross_val_results[['test_accuracy', 'test_f1_macro',
       'test_precision_macro', 'test_recall_macro']].mean().values
results_df

,Accuracy,F1 Macro,Precision Macro,Recall Macro
RF,0.621345,0.619199,0.625543,0.622222
GB,0.598487,0.596573,0.601339,0.598693
CLF,0.672605,0.668591,0.681769,0.672222
XGB,0.597815,0.592646,0.605552,0.599346


In [662]:
confusion(y_train, pd.Series(xgb.predict(X_train)))

predicted,0,1
target,,
0,87,0
1,1,86


In [663]:
confusion(y_test, pd.Series(xgb.predict(X_test)))

predicted,0,1
target,,
0,71,32
1,26,17


# AdaBoosting

In [664]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier


In [665]:
base = DecisionTreeClassifier(max_depth=1, class_weight='balanced')
ada = AdaBoostClassifier(estimator=base, n_estimators=100, random_state=42)
ada.fit(X_train, y_train)


C:\Users\haoka\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


AdaBoostClassifier(estimator=DecisionTreeClassifier(class_weight='balanced',
                                                    max_depth=1),
                   n_estimators=100, random_state=42)

In [666]:
cross_val_results = pd.DataFrame(cross_validate(ada , X_train, y_train, cv = 5, 
                            scoring = ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro'] ))

results_df.loc['ADA',:] = cross_val_results[['test_accuracy', 'test_f1_macro',
       'test_precision_macro', 'test_recall_macro']].mean().values
results_df

C:\Users\haoka\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
C:\Users\haoka\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
C:\Users\haoka\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent th

,Accuracy,F1 Macro,Precision Macro,Recall Macro
RF,0.621345,0.619199,0.625543,0.622222
GB,0.598487,0.596573,0.601339,0.598693
CLF,0.672605,0.668591,0.681769,0.672222
XGB,0.597815,0.592646,0.605552,0.599346
ADA,0.591429,0.589899,0.593735,0.592157


In [667]:
confusion(y_train, pd.Series(ada.predict(X_train)))

predicted,0,1
target,,
0,86,1
1,0,87


In [668]:
confusion(y_test, pd.Series(ada.predict(X_test)))

predicted,0,1
target,,
0,67,36
1,17,26


In [ ]:
rf_model = ExtraTreesClassifier(class_weight='balanced')

ntrees = [150, None]
max_depth = [100, None]
min_samples_split = [4, 6]
min_samples_leaf = [2, 4]
balance = [None, 'balanced', 'balanced_subsample']

trc = GridSearchCV(estimator=rf_model,
                   scoring=scoring_dict,
                   param_grid={
                       'n_estimators': ntrees,
                       'max_depth': max_depth,
                       'min_samples_split': min_samples_split,
                       'min_samples_leaf': min_samples_leaf,
                       'class_weight': balance
                   },
                   cv=5,
                   return_train_score=True,
                   refit=False,
                   n_jobs=-1)

model_5CV = trc.fit(X_train, y_train)
print(timedelta(seconds=(time() - init_time)))

# Final test export

In [669]:
ILDS_test = pd.read_csv("log_ILDS_test_X.csv", delimiter=',', header=None)

ILDS_test.columns = ['Age','TP','ALB','AR','DBratio','logTB','logDB','logAlkphos','logSgpt','logSgot','Female']

X_test = ILDS_test.loc[:,:'Female']

ILDS_test['Label'] = voting.predict(X_test)

ILDS_test.head()

ILDS_test.index = ILDS_test.index + 1
ILDS_test.index.name = 'ID'

ILDS_test['Label'].to_csv('random_forest.csv', index=True)